In [1]:
!pip install plotly
!pip install nbformat


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [3]:
df = pd.read_csv('../data/processed_smiley.xlsx')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 165478 entries, 0 to 165477
Data columns (total 18 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Unnamed: 0       165478 non-null  int64  
 1   navnelbnr        165478 non-null  int64  
 2   tjek_nr          165478 non-null  int64  
 3   By               162861 non-null  str    
 4   Geo_Lat          110267 non-null  float64
 5   Geo_Lng          110267 non-null  float64
 6   URL              165478 non-null  str    
 7   adresse1         165478 non-null  str    
 8   cvrnr            164884 non-null  float64
 9   navn1            165478 non-null  str    
 10  pnr              164675 non-null  float64
 11  postnr           165478 non-null  int64  
 12  virksomhedstype  165478 non-null  str    
 13  kontrol          165478 non-null  float64
 14  dato             165478 non-null  str    
 15  category_group   165478 non-null  str    
 16  is_preliminary   165478 non-null  bool   
 17  ca

In [4]:
branch_counts = (
    df.groupby("pnr")
      .size()
      .reset_index(name="n_branches")
)

print(branch_counts["n_branches"].describe())

# chains with many branches
print(
    branch_counts[branch_counts["n_branches"] >= 5]
    .sort_values("n_branches", ascending=False)
    .head(20)
)

count    43591.000000
mean         3.777729
std          4.237732
min          1.000000
25%          2.000000
50%          4.000000
75%          4.000000
max        231.000000
Name: n_branches, dtype: float64
                pnr  n_branches
6997   1.003352e+09         231
6404   1.003310e+09         202
7649   1.003399e+09         198
5502   1.003257e+09         188
5879   1.003275e+09         142
4062   1.003051e+09         135
2008   1.001762e+09         128
18     1.000018e+09         113
9786   1.007210e+09         101
5344   1.003252e+09         100
6989   1.003351e+09         100
3305   1.002894e+09          87
5545   1.003259e+09          87
6408   1.003310e+09          82
1970   1.001741e+09          74
6168   1.003290e+09          71
7254   1.003367e+09          70
6961   1.003351e+09          67
24374  1.022019e+09          66
6972   1.003351e+09          65


In [20]:
import pandas as pd
import re
from collections import Counter

# =====================================================
# ASSUMPTION:
# df already exists and contains:
# - navnelbnr
# - navn1
# =====================================================

# =====================================================
# STEP 1 — CLEAN TEXT
# =====================================================

def clean_name(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    # remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # remove numbers
    text = re.sub(r"\d+", " ", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# =====================================================
# STEP 2 — EXTRACT COMMON BRANCH NAME
# =====================================================

def extract_branch_name(names):

    cleaned = [clean_name(n) for n in names if pd.notna(n)]

    if len(cleaned) == 0:
        return None

    # tokenize each name
    token_lists = [set(name.split()) for name in cleaned]

    # -------------------------------------------------
    # Find words common across ALL names
    # -------------------------------------------------

    common_tokens = set.intersection(*token_lists)

    # generic words to ignore
    stopwords = {
        "vej",
        "gade",
        "torv",
        "center",
        "centret",
        "butik",
        "restaurant",
        "pizza",
        "cafe",
        "café",
        "aps",
        "shop",
        "store",
        "kiosk"
    }

    common_tokens = [
        token for token in common_tokens
        if len(token) > 2 and token not in stopwords
    ]

    # if we found shared meaningful words
    if len(common_tokens) > 0:

        # sort for stable output
        branch_name = " ".join(sorted(common_tokens))

        return branch_name.title()

    # -------------------------------------------------
    # FALLBACK:
    # Most common token or bigram
    # -------------------------------------------------

    all_candidates = []

    for name in cleaned:

        tokens = name.split()

        # single words
        all_candidates.extend(tokens)

        # two-word phrases
        bigrams = [
            tokens[i] + " " + tokens[i + 1]
            for i in range(len(tokens) - 1)
        ]

        all_candidates.extend(bigrams)

    counts = Counter(all_candidates)

    # keep only meaningful candidates
    candidates = [
        (token, count)
        for token, count in counts.items()
        if len(token) > 3
    ]

    if len(candidates) == 0:
        return None

    # choose most frequent
    best_candidate = sorted(
        candidates,
        key=lambda x: x[1],
        reverse=True
    )[0][0]

    return best_candidate.title()


# =====================================================
# STEP 3 — CREATE BRANCH NAME MAP
# =====================================================

branch_name_map = (
    df.groupby("pnr")["navn1"]
      .apply(lambda x: extract_branch_name(x.tolist()))
)

# =====================================================
# STEP 4 — ADD branch_name COLUMN
# =====================================================

df["branch_name"] = df["pnr"].map(branch_name_map)

# =====================================================
# STEP 5 — OPTIONAL CLEANUP
# =====================================================

# remove empty strings if any
df["branch_name"] = df["branch_name"].replace("", pd.NA)

# =====================================================
# RESULT
# =====================================================

print(df[[
    "pnr",
    "navn1",
    "branch_name"
]].head(20))

             pnr                            navn1  \
0   1.017699e+09                      Firdaws ApS   
1   1.001418e+09                    KonditorBager   
2   1.004954e+09               NYTORVS BAGERI APS   
3   1.019633e+09      SuperBrugsen Havdrup Bageri   
4   1.028920e+09                      Digebageren   
5   1.021167e+09                    TAFFELBAY ApS   
6   1.001386e+09  Bageri Ingeborg v/Tommy Carlsen   
7   1.018905e+09              Bakkegårdens Bageri   
8   1.021639e+09                 Munchies Cookies   
9   1.021813e+09                 Lagkagehuset A/S   
10  1.026607e+09                  Landlyst Bageri   
11  1.003078e+09  Kvickly Spinderiet Bageriudsalg   
12  1.031428e+09                   Søde øjeblikke   
13  1.011335e+09                 føtex 1388 Bager   
14  1.031017e+09                   Tata Creations   
15  1.030568e+09                  Stengers Surdej   
16  1.030723e+09         Bageriet De Skøre Brødre   
17           NaN                     Karins ka

In [25]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# ----------------------------
# Columns
# ----------------------------
pnr_col        = "pnr"
name_col       = "navn1"
score_col      = "kontrol"
branch_col     = "navnelbnr"
branch_name_col = "branch_name"

# ----------------------------
# Prepare data
# ----------------------------
plot_df = df[[pnr_col, name_col, branch_col, branch_name_col, score_col]].dropna(subset=[pnr_col, branch_col, score_col]).copy()

plot_df[pnr_col]    = pd.to_numeric(plot_df[pnr_col],    errors="coerce")
plot_df[score_col]  = pd.to_numeric(plot_df[score_col],  errors="coerce")
plot_df[branch_col] = pd.to_numeric(plot_df[branch_col], errors="coerce")
plot_df = plot_df.dropna(subset=[pnr_col, score_col, branch_col])

# Most common chain name per PNR (from navn1)
name_lookup = (
    plot_df.groupby(pnr_col)[name_col]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
    .reset_index()
    .rename(columns={name_col: "chain_name"})
)
plot_df = plot_df.merge(name_lookup, on=pnr_col, how="left")

# ----------------------------
# Aggregate: worst score per branch (navnelbnr)
# Also carry the branch_name through (most common per navnelbnr)
# ----------------------------
branch_df = (
    plot_df.groupby([pnr_col, "chain_name", branch_col])
    .agg(
        worst_score=(score_col, "max"),
        branch_name=(branch_name_col, lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
    )
    .reset_index()
)

# Top 12 chains by branch count
top_pnrs = (
    branch_df.groupby(pnr_col).size()
    .sort_values(ascending=False)
    .head(12).index
)
branch_df = branch_df[branch_df[pnr_col].isin(top_pnrs)]

branch_counts = branch_df.groupby("chain_name").size().rename("n_branches")

score_dist = (
    branch_df.groupby(["chain_name", "worst_score"])
    .size().unstack(fill_value=0)
)
score_pct = score_dist.div(score_dist.sum(axis=1), axis=0) * 100

for s in [1.0, 2.0, 3.0, 4.0]:
    if s not in score_pct.columns:
        score_pct[s] = 0.0

score_pct = score_pct.sort_values(by=1.0, ascending=True)
chain_order = score_pct.index.tolist()
n_chains = len(chain_order)

# ----------------------------
# Score metadata
# ----------------------------
SCORES = [1.0, 2.0, 3.0, 4.0]
COLORS = {1.0: "#2F8F4E", 2.0: "#C7A008", 3.0: "#C86D1F", 4.0: "#B7352D"}
LABELS = {1.0: "Best", 2.0: "Okay", 3.0: "Poor", 4.0: "Very poor"}

# ----------------------------
# Figure dimensions
# ----------------------------
LEFT_MARGIN  = 260
RIGHT_MARGIN = 60
TOP_MARGIN   = 200
BOT_MARGIN   = 80
ROW_H        = 58
FIG_W        = 1200
FIG_H        = TOP_MARGIN + BOT_MARGIN + n_chains * ROW_H

fig = go.Figure()

# ----------------------------
# Bars
# ----------------------------
for score in SCORES:
    pcts = score_pct[score].reindex(chain_order).fillna(0).values
    text_labels = [f"{v:.0f}%" if v >= 6 else "" for v in pcts]

    fig.add_trace(go.Bar(
        y=chain_order,
        x=pcts,
        name=LABELS[score],
        orientation="h",
        marker=dict(color=COLORS[score], line=dict(width=0)),
        text=text_labels,
        textposition="inside",
        insidetextanchor="middle",
        textfont=dict(size=13, color="white", family="Arial, sans-serif"),
        hovertemplate=(
            "<b>%{y}</b><br>"
            f"{LABELS[score]}<br>"
            "Share of branches: %{x:.1f}%<extra></extra>"
        ),
    ))

# ----------------------------
# Layout
# ----------------------------
fig.update_layout(
    barmode="stack",
    width=FIG_W,
    height=FIG_H,
    paper_bgcolor="#F3F0EA",
    plot_bgcolor="#F3F0EA",
    font=dict(family="Georgia, serif", size=13, color="#111"),
    margin=dict(l=LEFT_MARGIN, r=RIGHT_MARGIN, t=TOP_MARGIN, b=BOT_MARGIN),
    showlegend=False,
    xaxis=dict(
        range=[0, 100],
        tickvals=[0, 25, 50, 75, 100],
        ticktext=["0%", "25%", "50%", "75%", "100%"],
        showgrid=True,
        gridcolor="rgba(0,0,0,0.10)",
        zeroline=False,
        tickfont=dict(size=12, color="#666", family="Arial, sans-serif"),
        title=dict(text="Share of branches (%)", font=dict(size=12, color="#666", family="Arial, sans-serif")),
    ),
    yaxis=dict(
        autorange="reversed",
        showgrid=False,
        ticks="",
        showticklabels=False,
        categoryorder="array",
        categoryarray=chain_order,
    ),
    bargap=0.32,
)

# ----------------------------
# Helper: paper-y for row i (centred on bar)
# ----------------------------
def row_paper_y(i):
    plot_h = FIG_H - TOP_MARGIN - BOT_MARGIN
    # fraction from top of plot area
    frac = (i + 0.5) / n_chains
    # convert to paper coords
    return 1.0 - (frac * plot_h / FIG_H)

# ----------------------------
# Title block (absolute pixel feel via paper coords)
# Titles sit in the top margin — we anchor relative to paper y > 1
# ----------------------------
title_y_base = 1.0 + (TOP_MARGIN / FIG_H)   # very top of paper

fig.add_annotation(
    xref="paper", yref="paper",
    x=-(LEFT_MARGIN / (FIG_W - LEFT_MARGIN - RIGHT_MARGIN)),
    y=title_y_base - 0.010,
    text="5. CHAINS, BRANCHES & SYSTEMIC PATTERNS",
    showarrow=False, xanchor="left", yanchor="top",
    font=dict(family="Arial, sans-serif", size=11, color="#999", letterSpacing=2),
)
fig.add_annotation(
    xref="paper", yref="paper",
    x=-(LEFT_MARGIN / (FIG_W - LEFT_MARGIN - RIGHT_MARGIN)),
    y=title_y_base - 0.072,
    text="<b>Do restaurant chains share the same problems?</b>",
    showarrow=False, xanchor="left", yanchor="top",
    font=dict(family="Georgia, serif", size=30, color="#111"),
)
fig.add_annotation(
    xref="paper", yref="paper",
    x=-(LEFT_MARGIN / (FIG_W - LEFT_MARGIN - RIGHT_MARGIN)),
    y=title_y_base - 0.175,
    text="Each branch counted once — using its <i>worst</i> inspection score. Sorted by share of best-rated branches.",
    showarrow=False, xanchor="left", yanchor="top",
    font=dict(family="Georgia, serif", size=13, color="#666"),
)

# ----------------------------
# Legend  (sits just below subtitle, well above first bar)
# ----------------------------
LX = -(LEFT_MARGIN / (FIG_W - LEFT_MARGIN - RIGHT_MARGIN))  # left edge in paper x
legend_y = title_y_base - 0.265

fig.add_annotation(
    xref="paper", yref="paper",
    x=LX, y=legend_y,
    text="<b>Inspection score</b>",
    showarrow=False, xanchor="left", yanchor="middle",
    font=dict(family="Arial, sans-serif", size=13, color="#333"),
)

# Each legend item: coloured circle + number + label
# We space them in paper-x units; 1 paper-x unit = plot width in px
plot_w = FIG_W - LEFT_MARGIN - RIGHT_MARGIN
px_per_paper = plot_w   # 1 paper unit = plot_w px

item_gap_px = 105       # px between items
x_px = 145              # starting offset in px from left edge of plot area

for score, label in [(1.0,"Best"),(2.0,"Okay"),(3.0,"Poor"),(4.0,"Very poor")]:
    cx = LX + x_px / px_per_paper
    color = COLORS[score]
    # Big filled circle
    fig.add_annotation(
        xref="paper", yref="paper", x=cx, y=legend_y,
        text=f"<span style='color:{color}'>⬤</span>",
        showarrow=False, xanchor="center", yanchor="middle",
        font=dict(size=28),
    )
    # White number centred on circle
    fig.add_annotation(
        xref="paper", yref="paper", x=cx, y=legend_y,
        text=f"<b>{int(score)}</b>",
        showarrow=False, xanchor="center", yanchor="middle",
        font=dict(family="Arial, sans-serif", size=12, color="white"),
    )
    # Label to the right
    fig.add_annotation(
        xref="paper", yref="paper", x=cx + 18/px_per_paper, y=legend_y,
        text=label,
        showarrow=False, xanchor="left", yanchor="middle",
        font=dict(family="Arial, sans-serif", size=13, color="#333"),
    )
    x_px += item_gap_px

# ----------------------------
# Thin divider above first row
# ----------------------------
fig.add_shape(
    type="line", xref="paper", yref="paper",
    x0=LX, x1=1.0,
    y0=1.002, y1=1.002,
    line=dict(color="rgba(0,0,0,0.15)", width=1),
)

# ----------------------------
# Per-row labels: bold name + grey "N branches" subline
# ----------------------------
for i, chain in enumerate(chain_order):
    py = row_paper_y(i)
    n  = int(branch_counts.get(chain, 0))
    sub_offset = 0.5 / n_chains   # half a row height in paper units

    # Bold chain name — right-aligned just left of bar
    fig.add_annotation(
        xref="paper", yref="paper",
        x=-0.01, y=py + sub_offset * 0.35,
        text=f"<b>{chain}</b>",
        showarrow=False, xanchor="right", yanchor="bottom",
        font=dict(family="Georgia, serif", size=14, color="#111"),
    )
    # Grey branch count subline
    fig.add_annotation(
        xref="paper", yref="paper",
        x=-0.01, y=py - sub_offset * 0.35,
        text=f"{n} branches",
        showarrow=False, xanchor="right", yanchor="top",
        font=dict(family="Arial, sans-serif", size=11, color="#999"),
    )

# ----------------------------
# Footer
# ----------------------------
fig.add_annotation(
    xref="paper", yref="paper",
    x=LX, y=-(BOT_MARGIN / FIG_H) * 0.5,
    text="<i>Each branch counted once using its worst recorded inspection score. Branch names from branch_name column. Percentages may not sum to 100 due to rounding.</i>",
    showarrow=False, xanchor="left", yanchor="top",
    font=dict(family="Arial, sans-serif", size=11, color="#aaa"),
)

# ----------------------------
# Save
# ----------------------------
output_path = "chain_inspection_chart.html"
fig.write_html(output_path, include_plotlyjs="cdn", full_html=True)
print(f"Saved to {output_path}")

ValueError: Invalid property specified for object of type plotly.graph_objs.layout.annotation.Font: 'letterSpacing'

Did you mean "lineposition"?

    Valid properties:
        color

        family
            HTML font family - the typeface that will be applied by
            the web browser. The web browser can only apply a font
            if it is available on the system where it runs. Provide
            multiple font families, separated by commas, to
            indicate the order in which to apply fonts if they
            aren't available.
        lineposition
            Sets the kind of decoration line(s) with text, such as
            an "under", "over" or "through" as well as combinations
            e.g. "under+over", etc.
        shadow
            Sets the shape and color of the shadow behind text.
            "auto" places minimal shadow and applies contrast text
            font color. See https://developer.mozilla.org/en-
            US/docs/Web/CSS/text-shadow for a

In [8]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 165478 entries, 0 to 165477
Data columns (total 18 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Unnamed: 0       165478 non-null  int64  
 1   navnelbnr        165478 non-null  int64  
 2   tjek_nr          165478 non-null  int64  
 3   By               162861 non-null  str    
 4   Geo_Lat          110267 non-null  float64
 5   Geo_Lng          110267 non-null  float64
 6   URL              165478 non-null  str    
 7   adresse1         165478 non-null  str    
 8   cvrnr            164884 non-null  float64
 9   navn1            165478 non-null  str    
 10  pnr              164675 non-null  float64
 11  postnr           165478 non-null  int64  
 12  virksomhedstype  165478 non-null  str    
 13  kontrol          165478 non-null  float64
 14  dato             165478 non-null  str    
 15  category_group   165478 non-null  str    
 16  is_preliminary   165478 non-null  bool   
 17  ca

# Generate chain-branches.html for website

In [ ]:
import pandas as pd
import re
from collections import Counter

# most recent inspection per branch
df_cb = df.copy()
df_cb['dato'] = pd.to_datetime(df_cb['dato'], errors='coerce')
latest = df_cb.sort_values('dato').groupby('navnelbnr').last().reset_index()
latest = latest.dropna(subset=['pnr'])

SKIP_WORDS = {'og','af','til','med','for','den','det','aps','a/s','kiosk',
              'afd','afsnit','afsnitskøkken','centralkøkken','boder','indgang',
              'sal','kaffebar','mad','drikke','go','to','nord','syd','øst','vest',
              'pub','bar','shop','cafe','cafeteria','køkken','grill','pizza',
              'pasta','buffet','ice','slush','de','afs','afd.'}
EXCL_ABBREV = {'NaN','AS','APS','AFD','KS','IS','AB'}

def operator_name(names):
    names = [str(n) for n in names]
    total = len(names)
    abbrev_freq = Counter()
    for n in names:
        for token in n.split():
            t = re.sub(r'[^\w]', '', token)
            if t.isupper() and 3 <= len(t) <= 6 and t not in EXCL_ABBREV:
                abbrev_freq[t] += 1
    cands = [(t, c) for t, c in abbrev_freq.items() if c / total >= 0.70]
    if cands:
        return max(cands, key=lambda x: x[1])[0]
    cleaned = []
    for n in names:
        s = re.sub(r'^[\d\s\.\-]+', '', n)
        s = re.sub(r',.*$', '', s)
        s = re.sub(r'\s+-\s+.*$', '', s)
        s = re.sub(r'\baps\b|\ba/s\b', '', s, flags=re.IGNORECASE)
        s = ' '.join(w for w in s.split() if not re.search(r'\d', w)).strip()
        if len(s) >= 3:
            cleaned.append(s)
    if not cleaned:
        return str(names[0])[:20]
    freq2 = Counter()
    for name in cleaned:
        words = name.split()
        seen = set()
        for i in range(len(words) - 1):
            bg = f"{words[i].lower()} {words[i+1].lower()}"
            if bg not in seen:
                freq2[bg] += 1
                seen.add(bg)
    bigram_cands = [(bg, c) for bg, c in freq2.items()
                    if c / total >= 0.38
                    and not any(w.rstrip('.') in SKIP_WORDS for w in bg.split())]
    if bigram_cands:
        best_bg = max(bigram_cands, key=lambda x: (x[1], len(x[0])))[0]
        for name in cleaned:
            if best_bg in name.lower():
                idx = name.lower().index(best_bg)
                return name[idx:idx+len(best_bg)]
        return best_bg.title()
    freq1 = Counter()
    for name in cleaned:
        for w in set(name.split()):
            freq1[w.lower()] += 1
    uni_cands = [(w, c) for w, c in freq1.items()
                 if c / total >= 0.38 and w.rstrip('.') not in SKIP_WORDS and len(w) >= 4]
    if uni_cands:
        best_w = max(uni_cands, key=lambda x: (x[1], len(x[0])))[0]
        for name in cleaned:
            for token in name.split():
                if token.lower() == best_w:
                    return token
        return best_w.title()
    mc = Counter(cleaned).most_common(1)[0][0]
    return ' '.join(mc.split()[:2])

# top 12 chains by unique branch count (most recent inspection per branch)
chain_sizes = latest.groupby('pnr').size().sort_values(ascending=False).head(12)
top_pnrs    = chain_sizes.index.tolist()
sub         = latest[latest['pnr'].isin(top_pnrs)].copy()

rows = []
for pnr_val in top_pnrs:
    chain_data = sub[sub['pnr'] == pnr_val]
    n    = len(chain_data)
    name = operator_name(chain_data['navn1'].tolist())
    dist = chain_data['kontrol'].value_counts(normalize=True) * 100
    rows.append({
        'label':  f"{name} ({n} branches)",
        'score1': round(dist.get(1.0, 0), 1),
        'score2': round(dist.get(2.0, 0), 1),
    })

chain_df = pd.DataFrame(rows).sort_values('score1', ascending=True).reset_index(drop=True)
chain_df

In [ ]:
import re, json
import numpy as np

def inject(html, **kw):
    for k, v in kw.items():
        html = re.sub(rf'/\*{k}\*/.*?/\*END_{k}\*/',
                      lambda m, v=v: f'/*{k}*/{v}/*END_{k}*/', html, flags=re.DOTALL)
        html = re.sub(rf'<!--{k}-->.*?<!--/{k}-->',
                      lambda m, v=v: f'<!--{k}-->{v}<!--/{k}-->', html, flags=re.DOTALL)
    return html

class _NpEnc(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, np.integer): return int(o)
        if isinstance(o, np.floating): return float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        return super().default(o)

labels = chain_df['label'].tolist()
score1 = chain_df['score1'].tolist()
score2 = chain_df['score2'].tolist()

traces_js = json.dumps([
    {
        "type": "bar", "orientation": "h", "name": "No remarks",
        "x": score1, "y": labels,
        "marker": {"color": "rgba(34,197,94,0.75)", "line": {"width": 0}},
        "hovertemplate": "<b>%{y}</b><br>Score 1 — No remarks: %{x:.1f}%<extra></extra>",
        "showlegend": False,
    },
    {
        "type": "bar", "orientation": "h", "name": "Minor remarks",
        "x": score2, "y": labels,
        "marker": {"color": "rgba(234,179,8,0.75)", "line": {"width": 0}},
        "hovertemplate": "<b>%{y}</b><br>Score 2 — Minor remarks: %{x:.1f}%<extra></extra>",
        "showlegend": False,
    },
], cls=_NpEnc, ensure_ascii=False)

total_count   = len(chain_df)
perfect_count = int((chain_df['score1'] == 100.0).sum())
worst_row     = chain_df.loc[chain_df['score2'].idxmax()]
worst_full    = worst_row['label']
worst_pct     = worst_row['score2']
worst_branches = re.search(r'\((\d+) branches\)', worst_full).group(1)
worst_name    = re.sub(r'\s*\(\d+ branches\)$', '', worst_full)

tmpl = open('../figures/chain-branches.html').read()
out  = inject(tmpl,
    TRACES_DATA  = traces_js,
    TOTAL_COUNT  = str(total_count),
    PERFECT_BOX  = (f'<span class="cb-hl green">{perfect_count} of {total_count}</span> large operators had a 100% clean '
                    f'record across every branch.'),
    HIGH_BAR_BOX = (f'All {total_count} operators score <span class="cb-hl blue">95%+</span> on their most '
                    f'recent inspection round. Size and systemic processes appear to drive '
                    f'consistently high compliance.'),
    WORST_BOX    = (f'{worst_name} had the most minor remarks &mdash; '
                    f'<span class="cb-hl yellow">{worst_pct:.1f}% of its {worst_branches} branches</span> received a '
                    f'Score 2 on their latest visit. No serious problems were found.'),
)
open('../figures/chain-branches.html', 'w').write(out)
open('../sections/chain-branches.html', 'w').write(out)
print(f'Done. {perfect_count}/{total_count} perfect | worst={worst_name} ({worst_pct:.1f}%, {worst_branches} branches)')